# Semantic Chunking for RAG via Deep Learning (BiLSTM)

Replace fixed-size chunking with a **learned semantic boundary detector** (BiLSTM),
trained on Wikipedia section pseudo-labels, and evaluate retrieval quality on
**Natural Questions**.

**How to run this notebook**
1. Upload the whole `RAG chunk optimize` folder to your Google Drive (e.g. `MyDrive/RAG chunk optimize`).
2. Runtime -> Change runtime type -> **GPU** (T4 is plenty).
3. Run the cells top-to-bottom. Every phase caches to Drive, so you can stop / resume any time.

All tunables live in `config.py` (article count, NQ doc count, hyper-params).

## 0. Setup

In [ ]:
# torch is pre-installed on Colab; install the rest.
!pip -q install datasets sentence-transformers faiss-cpu nltk mwparserfromhell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
# Folder you uploaded (the project root that contains config.py). Edit if needed.
PROJECT_DIR = '<project-root>'
# Where big intermediate artifacts are cached (kept next to the code on Drive).
os.environ['RAG_DATA_ROOT'] = PROJECT_DIR + '/artifacts'
sys.path.insert(0, PROJECT_DIR)
# Stand in the project root so later `!python scripts/...` cells find scripts/.
os.chdir(PROJECT_DIR)

import config as C
C.ensure_dirs()
print(C.summary())

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())

## Smoke test (recommended first run)

Runs all 5 phases at tiny scale (~30 wiki articles, 1 epoch, ~10 NQ docs) in a separate `*_smoke` folder to confirm the pipeline works end-to-end **before** the full run. The printed numbers are meaningless (tiny data) — this only checks that every phase wires together (including the NQ download). Real config is restored automatically afterwards, so you can run the full phases below in the same kernel.

In [ ]:
from rag_chunk import smoke
_ = smoke.run_smoke()

## Phase 1 - Data preparation (Wikipedia -> sentences + boundary labels)

Picks article titles by streaming the HF dump, fetches **raw wikitext** via the
MediaWiki API (batched, cached, resumable), parses real `== Section ==` structure
with `mwparserfromhell`, and writes `train/val/test.jsonl`.

In [ ]:
from rag_chunk import wiki_data
stats = wiki_data.prepare_dataset(C.N_WIKI_ARTICLES)
stats

## Phase 2 - Offline embedding (all-MiniLM-L6-v2)

Caches one `(n_sentences, 384)` array per article to Drive. Resumable.

In [ ]:
from rag_chunk import embedding
embedding.embed_offline()

## Phase 3 - Train the BiLSTM boundary detector

Weighted BCE (`pos_weight = #neg/#pos`), one article per step, early stopping on
validation loss. Best weights are saved to Drive.

In [ ]:
from rag_chunk import training
train_stats = training.train_model()
train_stats['test_boundary_f1']

## Phase 4 - Build RAG indices on Natural Questions

Streams NQ `validation`, chunks each document two ways (BiLSTM vs fixed 5-sentence),
embeds the chunks, and builds two FAISS `IndexFlatIP` indices.

In [ ]:
from rag_chunk import retrieval, training
model = training.load_model()
built = retrieval.build_indexes(model)
print('bilstm chunks:', len(built['bilstm'].chunk_texts),
      '| fixed chunks:', len(built['fixed'].chunk_texts),
      '| questions:', len(built['questions']))

## Phase 5 - Evaluate (Recall@k + Boundary F1)

Prints the comparison table and writes `results/recall_comparison.csv` and `.png`.

In [ ]:
from rag_chunk import evaluation, training
model = training.load_model()
results = evaluation.evaluate_all(model)
results

In [ ]:
from IPython.display import Image
Image(filename=str(C.RESULTS_FIGURE))

## Phase 6 - Chunking sweep optimizer

Sweeps **fixed-size** and learned **target-size** chunking across chunk sizes and
overlaps, scores each on NQ doc-constrained Recall@k, and exports optimizer
artifacts to `artifacts/results/latest/`: `sweep_results.csv`, `best_config.json`,
`fair_comparison_table.csv`, `recall_vs_chunk_size.png`.

This stage does **not** assume learned chunking always wins — it controls for chunk
size and overlap, then measures which strategy retrieves answer-bearing chunks
better. Per-config FAISS indices are built in memory and discarded (nothing is
cached under `nq/` unless you pass `--save-sweep-index`).

In [ ]:
from rag_chunk import sweep, training
model = training.load_model()
# Fast grid first (quick=True). For the full grid use quick=False, or run
# `python scripts/6_sweep_chunking.py`. run_sweep prints the ranked table + best config.
rows = sweep.run_sweep(model, quick=True)
print(f"\n{len(rows)} configs swept -> artifacts/results/latest/")

In [ ]:
from IPython.display import Image
Image(filename=str(C.RESULTS_LATEST_DIR / C.RECALL_PLOT_PNG))

## Phase 7 - Train Transformer boundary model (Stage 2)

Trains a 2-layer Transformer boundary detector as a **second** learned
chunking model. It reuses the **same** cached MiniLM sentence embeddings and
Wikipedia section labels from Phases 1-3 — **no need to rerun Phase 1/2**.
Best weights are saved to `models/transformer_best.pt`; the BiLSTM weights
(`models/bilstm_best.pt`) are left untouched.

In [ ]:
from rag_chunk import training
tf_stats = training.train_model(model_type="transformer")
tf_stats['test_boundary_f1']

## Phase 8 - Compare Fixed vs BiLSTM vs Transformer (Stage 2)

Runs the Phase 6 sweep with the Transformer added as a third method, under the
same MiniLM embeddings, the same cached NQ docs/questions, the same Recall@k
metric and the same target-size/overlap policy — so only the boundary model
differs. Writes `sweep_results.csv`, `best_config.json`,
`fair_comparison_table.csv`, `recall_vs_chunk_size.png` and
`model_comparison.png` to `artifacts/results/latest/`.

The cell below uses `quick=True` — a fast sanity grid `{8, 10, 12}` just to preview
the chart. **Do not archive this quick run as the Stage 3 baseline.** The Stage 3
section below has a dedicated cell that runs the *full-grid* Stage 2 sweep and
archives it, so the MiniLM-vs-BGE comparison stays matched at every chunk size.

In [ ]:
from rag_chunk import sweep, training
bilstm = training.load_model("bilstm")
transformer = training.load_model("transformer")
# quick=True for a fast grid; drop it (or run scripts/8_sweep_with_transformer.py) for the full grid.
rows = sweep.run_sweep(bilstm, transformer_model=transformer, quick=True)
print(f"\n{len(rows)} configs swept (fixed + bilstm + transformer) -> artifacts/results/latest/")

In [ ]:
from IPython.display import Image
Image(filename=str(C.RESULTS_LATEST_DIR / C.RECALL_PLOT_PNG))

## Stage 3 - BGE retrieval embedding ablation

This stage keeps the existing MiniLM boundary/chunking embeddings and the existing BiLSTM/Transformer weights. It only switches FAISS chunk embeddings and query embeddings to BGE.

Before running Stage 3, archive the completed Stage 2 latest results so this stage can safely overwrite `artifacts/results/latest/`.

⚠️ **The Stage 2 archive used as the Stage 3 baseline must be a full-grid sweep — not the `quick=True` Phase 8 cell.** Stage 3 sweeps the full grid `{6, 8, 10, 12, 15}`; if the archived Stage 2 baseline only has the quick grid `{8, 10, 12}`, the matched MiniLM-vs-BGE table is blank (NaN) at the sizes only Stage 3 has — including the headline `size=15`. The archive cell below runs the full grid explicitly, so run it (not the quick Phase 8 cell) before Stage 3.

In [ ]:
# Official Stage 2 baseline for Stage 3: run the FULL grid (no --quick), then archive it.
# This overwrites artifacts/results/latest/ with the full-grid Stage 2 sweep and copies
# it to artifacts/results/stage2/final/ so the MiniLM baseline is matched to Stage 3 at
# every chunk size (including size=15). Do NOT archive the quick Phase 8 cell instead.
!python scripts/8_sweep_with_transformer.py
!python scripts/save_stage_results.py --stage stage2

In [ ]:
# Full Stage 3 BGE retrieval sweep.
# If Colab is slow, switch to:
# !python scripts/9_sweep_bge_retrieval.py --retrieval-model BAAI/bge-small-en-v1.5
!python scripts/9_sweep_bge_retrieval.py


In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.SWEEP_RESULTS_CSV,
    C.BEST_CONFIG_JSON,
    C.FAIR_TABLE_CSV,
    'stage2_vs_stage3_matched.csv',
    C.RECALL_PLOT_PNG,
    C.MODEL_PLOT_PNG,
    'stage3_bge_retrieval_summary.md',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(Image(filename=str(latest / 'recall_vs_chunk_size.png')))
display(Image(filename=str(latest / 'model_comparison.png')))

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / 'sweep_results.csv'))
display(pd.read_csv(latest / 'fair_comparison_table.csv'))

## Stage 4 - Hybrid retrieval ablation (BM25 + BGE + RRF)

Same chunks, three retrievers per chunking config: **`bge`** (dense, exactly the Stage 3 path), **`bm25`** (lexical Okapi BM25 — pure numpy, no new install), and **`rrf`** (Reciprocal Rank Fusion of the two rankings, k=60, depth 50). Dataset size, chunking grids, boundary models/weights and the BGE model are all unchanged from Stage 3.

Run order:

1. **Archive Stage 3** (copy-only). Stage 4 refuses to start without `artifacts/results/stage3/final/`, and it only clears files from `latest/` that have a byte-identical archived copy — so nothing unarchived is ever deleted and `stage3/final` is never touched.
2. **Run the sweep.** It re-runs the BGE arm and compares it row-by-row against the archived Stage 3 baseline (`stage3_vs_stage4_bge_check.csv`): every `delta_n_chunks` must be 0 and every recall delta 0.0000 — the console prints **"check OK"** when the baseline is reproduced exactly.
3. **Only after the check passes**, archive Stage 4 to `artifacts/results/stage4/final/`.

In [ ]:
# Stage 4 step 1: archive Stage 3 (copy-only, does NOT rerun anything).
# Stage 4 refuses to run until artifacts/results/stage3/final/ exists.
!python scripts/save_stage_results.py --stage stage3

In [ ]:
# Stage 4 step 2: full hybrid sweep (BGE / BM25 / RRF over identical chunks).
# GPU cost is about the same as the Stage 3 sweep (dense embedding dominates;
# BM25 + RRF add seconds). No new pip installs needed.
!python scripts/10_sweep_hybrid_retrieval.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.HYBRID_SWEEP_CSV,
    C.HYBRID_BEST_JSON,
    C.HYBRID_MATCHED_CSV,
    'stage3_vs_stage4_bge_check.csv',
    C.HYBRID_SCATTER_PNG,
    C.HYBRID_RETRIEVER_PLOT_PNG,
    'stage4_hybrid_retrieval_summary.md',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(Image(filename=str(latest / C.HYBRID_SCATTER_PNG)))
display(Image(filename=str(latest / C.HYBRID_RETRIEVER_PLOT_PNG)))

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.HYBRID_MATCHED_CSV))
display(pd.read_csv(latest / 'stage3_vs_stage4_bge_check.csv'))

In [ ]:
# Stage 4 step 3: run ONLY after the sweep above printed "check OK"
# (the bge arm reproduces the archived Stage 3 baseline exactly).
!python scripts/save_stage_results.py --stage stage4

## Stage 5 - Cross-encoder reranking (BGE top-k + bge-reranker-base)

Same chunks, three arms per chunking config: **`bge`** (dense-only, exactly the Stage 3/4 path), **`rerank20`** (BGE top-20 candidates reordered by the pretrained cross-encoder `BAAI/bge-reranker-base`), and **`rerank50`** (the same with a top-50 pool). Dataset size, chunking grids, boundary models/weights and the BGE dense retriever are all unchanged; the reranker is off-the-shelf — **no training or fine-tuning**. Goal metric: **Recall@1 / Recall@3** (Recall@5 is already near its ceiling).

Every row also reports `pool_recall@{20,50}` — whether the answer chunk was in the BGE candidate pool at all. Reranking can only promote chunks the pool already contains, so read any improvement against that ceiling.

Run order:

1. **Run the sweep.** No archive step needed first — Stage 4 step 3 above already archived `latest/` to `stage4/final/`, and the script refuses to delete anything from `latest/` without a byte-identical archived copy (`stage3/final` and `stage4/final` are only ever read). It re-runs the BGE arm and compares it row-by-row against the archived Stage 3 baseline (`stage3_vs_stage5_bge_check.csv`) — the console prints **"check OK"** when the baseline is reproduced exactly.
2. **Only after the check passes**, archive Stage 5 to `artifacts/results/stage5/final/`.

In [ ]:
# Stage 5 step 1: full rerank sweep (bge / rerank20 / rerank50 over identical
# chunks). Downloads BAAI/bge-reranker-base on first run (no new pip installs —
# sentence-transformers CrossEncoder is already available). Reranking scores
# each (question, chunk) pair once, so the added GPU cost is roughly
# 30 configs x 203 questions x 50 pairs on top of the dense embedding.
# Requires artifacts/results/stage3/final/ AND everything in results/latest/
# already archived (Stage 4 step 3 above) — it aborts safely otherwise.
!python scripts/11_sweep_reranker.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.RERANK_SWEEP_CSV,
    C.RERANK_BEST_JSON,
    C.RERANK_MATCHED_CSV,
    'stage3_vs_stage5_bge_check.csv',
    C.RERANK_SCATTER_PNG,
    C.RERANK_COMPARISON_PNG,
    'stage5_reranker_summary.md',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(Image(filename=str(latest / C.RERANK_SCATTER_PNG)))
display(Image(filename=str(latest / C.RERANK_COMPARISON_PNG)))

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.RERANK_MATCHED_CSV))
display(pd.read_csv(latest / 'stage3_vs_stage5_bge_check.csv'))

In [ ]:
# Stage 5 step 2: run ONLY after the sweep above printed "check OK"
# (the bge arm reproduces the archived Stage 3 baseline exactly).
!python scripts/save_stage_results.py --stage stage5

## Stage 6 - Larger-scale robustness evaluation (~1000 docs)

Same dataset source, same chunking grids, same boundary models/weights, same BGE retriever, same off-the-shelf reranker (**top-20 only** — no rerank50, no fine-tuning). The only change: corpus scale **200 → ~1000 docs/questions**. Every stage so far ran at n=203 questions where 1 SE ≈ 0.034; at n≈1000 the SE halves, so the Stage 1-5 conclusions either firm up or get overturned — the actual loaded doc/question counts are printed and reported (the NQ stream stops at the N-th usable document, so they are not exactly 1000).

Arms: **`bge`** on the full 30-config grid; **`rerank20`** only on 5 selected configs (fixed 6/0, fixed 15/0, fixed 15/1, bilstm 15/0, transformer 15/0) — the small-chunk config plus the size-15 sweet spot, exactly what the direction claims below need.

Run order:

1. **Check mode first.** Re-runs everything at N=200 on the same cached corpus and must reproduce the archived Stage 5 rows **exactly** (`stage6_check_vs_stage5.csv`) — wait for **"check OK"**. This proves the Stage 6 code path *is* the Stage 3/5 pipeline before any large-scale conclusion is drawn.
2. **The large run.** The bigger corpus caches under a separate `nq/large_n1000/` folder, so the 200-doc cache is untouched. The eval set changes, so no exact-delta check is possible — instead `stage6_direction_check.csv` re-tests the four Stage 1-5 direction claims (size > method; BGE strong; rerank20 helps mainly small chunks; no clear gain at the size-15 sweet spot) with explicit rules and observed values.
3. **Archive** to `artifacts/results/stage6/final/` after reviewing the direction checks. Stage 3/4/5 finals are only ever read.

In [ ]:
# Stage 6 step 1: sanity mode at N=200 (the same cached corpus as Stages 3/5).
# Every row — bge on all 30 configs + rerank20 on the 5 selected configs —
# must reproduce the archived Stage 5 rows exactly: wait for "check OK".
# Requires artifacts/results/stage5/final/ (Stage 5 step 2 above).
# Resume-safe: if interrupted, re-run this same cell.
!python scripts/12_large_eval.py --check

In [ ]:
# Stage 6 step 2: the large eval (~1000 docs/questions; the actual loaded
# counts are printed and written into every output). Builds a separate NQ
# cache (nq/large_n1000/) on first run — the 200-doc cache is never touched.
# RESUME-SAFE: every finished config is checkpointed to
# latest/stage6_checkpoint_large.jsonl, so if the session dies just re-run
# this same cell and it continues where it stopped (--fresh starts over).
# Expect a few hours on a T4: ~30 configs x ~5x the Stage 3 embedding cost,
# plus rerank20 on 5 configs (~5 x 1000 questions x 20 pairs).
!python scripts/12_large_eval.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.STAGE6_RESULTS_CSV,
    C.STAGE6_MATCHED_CSV,
    C.STAGE6_DIRECTION_CSV,
    C.STAGE6_SUMMARY_MD,
    'stage6_check_vs_stage5.csv',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
import pandas as pd
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.STAGE6_MATCHED_CSV))
display(pd.read_csv(latest / C.STAGE6_DIRECTION_CSV))
display(pd.read_csv(latest / 'stage6_check_vs_stage5.csv'))

In [ ]:
# Stage 6 figures (run AFTER the large eval, BEFORE the archive cell below).
# Reads the stage6 CSVs in latest/ plus the archived Stage 5 CSVs and writes
# 2 PNGs into latest/ so they get archived together with the CSVs:
#   stage6_size_vs_recall.png - bge R@5 vs chunk size, 203 vs 1032 questions
#   stage6_rerank_delta.png   - rerank20-bge dR@1 on the 5 matched configs
!python scripts/13_stage6_plots.py

from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
display(Image(filename=str(latest / C.STAGE6_SIZE_PLOT_PNG)))
display(Image(filename=str(latest / C.STAGE6_DELTA_PLOT_PNG)))

In [ ]:
# Stage 6 step 3: run ONLY after the check-mode run printed "check OK", the
# large run finished (review the direction checks above first), and the plots
# cell above ran — so the two PNGs are archived together with the CSVs.
!python scripts/save_stage_results.py --stage stage6

## Stage 7 - Cross-dataset robustness check (TriviaQA rc.wikipedia)

Same chunking grids, same boundary models/weights, same BGE retriever — the **only** change is the QA dataset: NQ → TriviaQA `rc.wikipedia`, whose full Wikipedia entity pages are bundled in the dataset (no fetching). **bge arm only** (no BM25/RRF, no reranking, no fine-tuning). Gold documents = the question's entity pages whose text contains the answer string (distant supervision — weaker than NQ's annotated gold; every output says so). See `docs/stage7_cross_dataset.md`.

Run order:

1. **Check mode first.** Re-runs the 30-config bge-only sweep on the cached NQ 200-doc corpus; every row must reproduce the archived Stage 3 rows **exactly** (`stage7_check_vs_stage3.csv`) — wait for **"check OK"**. This also proves the multi-gold metric extension didn't change single-gold behaviour.
2. **The TriviaQA run.** Streams ~300 kept questions (the loader prints filter statistics and aborts if the corpus would make the chunk-size sweep degenerate). No exact-delta check is possible across datasets — `stage7_direction_check.csv` re-tests the size-vs-method claims with explicit rules instead.
3. **Archive** to `artifacts/results/stage7/final/` after reviewing the direction checks. Stage 1-6 finals are only ever read.

In [ ]:
# Stage 7 step 1: sanity mode on the cached NQ 200-doc corpus. All 30 bge
# configs must reproduce the archived Stage 3 rows exactly: wait for "check OK".
# Requires artifacts/results/stage3/final/. Resume-safe: re-run this same cell.
!python scripts/15_cross_dataset_eval.py --check

In [ ]:
# Stage 7 step 2: the TriviaQA rc.wikipedia eval (~300 kept questions; the
# actual doc/question counts and loader filter stats are printed and written
# into every output). The corpus caches under data/triviaqa/n300/ — the NQ
# caches are never touched.
# RESUME-SAFE: every finished config is checkpointed to
# latest/stage7_checkpoint_trivia.jsonl; if the session dies re-run this same
# cell (--fresh starts over). Expect roughly Stage 3-like cost on a T4
# (30 bge-only configs; corpus size depends on the kept questions' pages).
!python scripts/15_cross_dataset_eval.py

In [ ]:
from pathlib import Path
import config as C

latest = Path(C.RESULTS_LATEST_DIR)
expected = [
    C.STAGE7_RESULTS_CSV,
    C.STAGE7_MATCHED_CSV,
    C.STAGE7_DIRECTION_CSV,
    C.STAGE7_SUMMARY_MD,
    C.STAGE7_SCATTER_PNG,
    'stage7_check_vs_stage3.csv',
]
for name in expected:
    path = latest / name
    print(f'{name}:', 'OK' if path.exists() else 'MISSING', path)

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

display(pd.read_csv(latest / C.STAGE7_MATCHED_CSV))
display(pd.read_csv(latest / C.STAGE7_DIRECTION_CSV))
display(pd.read_csv(latest / 'stage7_check_vs_stage3.csv'))
display(Image(filename=str(latest / C.STAGE7_SCATTER_PNG)))

In [ ]:
# Stage 7 step 3: run ONLY after the check-mode run printed "check OK" and
# the TriviaQA run finished (review the direction checks above first).
!python scripts/save_stage_results.py --stage stage7

## Stage 8 - Fine-tune the cross-encoder reranker (Route C)

Trigger (Stage 6): at the size-15 sweet spot `pool_recall@20` ≈ 0.96 but R@1 ≈ 0.63 and the **off-the-shelf** reranker adds ~0 — ranking, not pool recall, is the remaining bottleneck. Stage 8 fine-tunes `BAAI/bge-reranker-base` on the NQ **train** split (every eval bench uses the validation split, so train/eval stay disjoint). See `docs/stage8_reranker_finetune.md`.

Run order (**two sessions, with a human decision in between**):

1. **Build data** — stream the train split: training corpus (2000 docs) + dev bench (the next 400 docs), mine (1 positive + 7 hard negatives) groups from each question's BGE top-20 pool at the deployment chunking (fixed 15/0).
2. **Train** — listwise cross-entropy, fp16, ~1–1.5 h on a T4; saves each epoch to `models/bge_reranker_ft/`.
3. **Go/no-go on the dev bench** — dev ΔR@1 (ft − off-the-shelf) at fixed 15/0: ≥ +0.02 → **GO**; ≤ 0 → **NO-GO, STOP** (archive the honest negative result); in between → at most one retry. The dev bench decides; it never produces claims.
4. **Final eval (ONLY if GO)** — the Stage 6 bench (1032 questions, 5 configs), three arms sharing one identical top-20 pool. Built-in check: the `bge` + `rerank20` rows must reproduce `stage6/final` **exactly** before the `rerank20_ft` rows mean anything.
5. **Archive** to `artifacts/results/stage8/final/`.

In [ ]:
# Stage 8 step 1: build the fine-tuning data (NQ TRAIN split — disjoint from
# every eval bench). Streams ~2400 usable docs, chunks the training corpus at
# fixed 15/0, mines (1 pos + 7 hard negatives) groups from the BGE top-20.
# Caches under data/nq_train/; re-running skips finished caches.
!python scripts/16_build_rerank_train_data.py

In [ ]:
# Stage 8 step 2: fine-tune BAAI/bge-reranker-base (listwise CE, fp16,
# 2 epochs, ~1-1.5 h on a T4). Saves each epoch to models/bge_reranker_ft/;
# if the session dies mid-training, re-run with
#   --init-model "$RAG_DATA_ROOT/models/bge_reranker_ft/epoch1" --epochs 1
!python scripts/17_train_reranker.py

In [ ]:
# Stage 8 step 3: GO/NO-GO gate on the held-out dev bench (~15 min).
# Three arms share one identical BGE top-20 pool: bge / rerank20 (off-the-
# shelf) / rerank20_ft. Read the verdict printed at the end:
#   GO        -> run step 4 below.
#   NO-GO     -> STOP. Do NOT run step 4; archive the negative result
#                (that is the Stage 8 finding) and report back.
#   GRAY-ZONE -> at most ONE retry (more data/epochs), then decide.
!python scripts/18_eval_reranker_ft.py --dev

In [ ]:
# Stage 8 step 4: final eval on the Stage 6 bench — RUN ONLY AFTER A "GO"
# VERDICT ABOVE. 5 configs x 3 arms on 1032 questions (~2 h on a T4;
# checkpointed, re-run this same cell to resume). The bge + rerank20 rows
# must reproduce stage6/final exactly ("check OK") — only then do the
# rerank20_ft rows count.
!python scripts/18_eval_reranker_ft.py

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import Image, display
import config as C

latest = Path(C.RESULTS_LATEST_DIR)

for name in [C.STAGE8_DEV_RESULTS_CSV, C.STAGE8_RESULTS_CSV,
             C.STAGE8_MATCHED_CSV, C.STAGE8_CHECK_CSV]:
    p = latest / name
    if p.exists():
        print(name)
        display(pd.read_csv(p))
if (latest / C.STAGE8_DELTA_PNG).exists():
    display(Image(filename=str(latest / C.STAGE8_DELTA_PNG)))

In [ ]:
# Stage 8 step 5: archive. Two legitimate archive points:
#   - after a NO-GO: archives the dev results + gate verdict (honest negative);
#   - after a GO + final eval: archives the full Stage 6-bench comparison
#     (only if the built-in check printed "check OK").
!python scripts/save_stage_results.py --stage stage8

## Route D - Interactive demo (Gradio)

One question, two chunking strategies side by side (**fixed 15/0** vs **BiLSTM t15/0**) on the Stage 6 bench (1000 docs / 1032 questions), ranked by one of three arms sharing the same BGE top-20 pool: `bge` / `rerank20` (off-the-shelf) / `rerank20_ft` (the Stage 8 fine-tuned model). Bench questions highlight the answer, badge gold-document chunks, and show each chunk's dense-rank movement after reranking.

- Runs **no experiments** and writes **nothing** under `results/` — safe next to the archives.
- First run builds + caches two demo FAISS indices under `data/nq/large_n1000/indices/demo/` (embeds ~40k chunks, a few minutes on a T4); later runs load them instantly.
- Click the public `*.gradio.live` link printed below; interrupt the cell to stop the server.

In [ ]:
# Route D: interactive demo (safe: read-only over the cached bench).
%pip install -q gradio
!python scripts/19_demo.py --share